In [1]:
from pathlib import Path
import pandas as pd

from phd_visualizations.regression import regression_plot
from phd_visualizations.calculations import SupportedInstruments 
from phd_visualizations import save_figure

import combined_cooler
import matlab

%reload_ext autoreload
%autoreload 2

data_path = Path("../data")

filename = "sc_exp1_exp.csv"

df = pd.read_csv(data_path / filename)
df


,Tv,qc,Tc_in,Tcond,Tc_out,mv
0,36.11,10.00,17.07,34.74,32.27,268.20
1,36.17,24.00,26.74,34.90,32.60,250.33
2,54.90,10.00,38.09,54.42,53.05,270.60
3,55.70,23.81,47.41,53.13,53.40,253.50
4,35.95,16.00,26.10,34.76,32.48,186.50
5,36.31,16.00,20.11,35.04,31.18,317.60
6,55.20,16.00,44.63,53.21,51.91,196.00
7,55.87,16.00,43.45,54.24,53.64,327.72
8,43.69,10.00,31.13,43.16,42.11,194.82
9,44.51,10.00,25.07,43.76,42.00,304.18


### Evaluate model

In [24]:
cc_model = combined_cooler.initialize()

results: list[dict] = []
Tc_in = []
Tc_out = []
for idx, ds in df.iterrows():

    mv_kgs = matlab.double([ds["mv"]/3600])  # Convert from kg/h to kg/s
    mc_kgs = matlab.double([ds["qc"]/3.6])  # Convert from m3/h to kg/s (assuming water density ~1000 kg/m3)
    Tv_C = matlab.double([ds["Tv"]])

    # Create the 'options' struct as a Python dictionary
    # options = {
    #     'model_type': 'data',
    #     'parameters': {},  # Empty struct in MATLAB
    #     'x0': float('nan'),  # Optional input as NaN
    #     'lb': matlab.double([43.324458513828800]),
    #     'ub': matlab.double([43.324458513828800]),
    # }

    tc_in, tc_out = cc_model.condenser_model(mv_kgs, Tv_C, mc_kgs, nargout=2)
    Tc_in.append(tc_in)
    Tc_out.append(tc_out)
    
results_df = pd.DataFrame({
    "Tv": df["Tv"],
    "Tc_in": Tc_in,
    "Tc_out": Tc_out,
    "qc": df["qc"],
    "mv": df["mv"],
    "Tcond": df["Tv"]
})
results_df


,Tv,Tc_in,Tc_out,qc,mv,Tcond
0,36.11,21.803732,37.306268,10.00,268.20,36.11
1,36.17,26.097781,32.126439,24.00,250.33,36.17
2,54.90,37.365255,52.706281,10.00,270.60,54.90
3,55.70,47.521408,53.700000,23.81,253.50,55.70
4,35.95,26.819283,33.557881,16.00,186.50,35.95
5,36.31,23.919240,35.390760,16.00,317.60,36.31
6,55.20,46.210807,53.200000,16.00,196.00,55.20
7,55.87,41.921376,53.521011,16.00,327.72,55.87
8,43.69,30.488109,41.690000,10.00,194.82,43.69
9,44.51,25.073786,42.436214,10.00,304.18,44.51


In [25]:
output_path = data_path / filename.replace("exp.csv", "mod_physical.csv")
results_df.to_csv(output_path)

print(f"Results saved to {output_path}")


Results saved to ../data/sc_exp1_mod_physical.csv


### Take from pre-evaluated

In [2]:
df = pd.read_csv(data_path / "sc_out_exp.csv")
results_df = pd.read_csv(data_path / "sc_out_mod.csv")
df_cal = pd.read_csv(data_path / "sc_exp1_exp.csv")
results_cal_df = pd.read_csv(data_path / "sc_exp1_mod_physical.csv")

df["subcooling"] = df["Tv"] - df["Tcond"]

display(df)
display(results_df)
display(df_cal)
display(results_cal_df)


,Date,qc,Tc_in,Tcond,Tc_out,mv,Tv,Tamb,Cw,subcooling
0,09-Feb-2024,16.00,25.13,34.98,32.39,205.61,36.03,14.89,88.248,1.05
1,26-Feb-2024,10.00,17.07,34.73,32.30,266.93,36.09,16.70,120.600,1.36
2,16-Apr-2024,22.43,33.12,41.59,38.24,212.22,43.30,25.06,0.000,1.71
3,16-Apr-2024,24.00,33.18,41.71,37.90,209.40,43.35,27.42,140.064,1.64
4,16-Apr-2024,23.74,33.16,41.92,38.53,235.36,43.52,29.84,124.710,1.60
5,16-Apr-2024,24.00,31.66,41.49,36.80,219.49,43.44,31.02,139.218,1.95
6,18-Apr-2024,24.00,33.17,41.67,38.25,231.26,43.28,19.55,124.800,1.61
7,18-Apr-2024,24.00,33.16,41.71,38.11,216.70,43.28,20.82,130.824,1.57
8,18-Apr-2024,24.00,33.90,42.88,40.20,269.14,43.55,22.24,0.000,0.67
9,18-Apr-2024,24.00,33.68,41.99,38.83,221.89,43.43,23.09,0.000,1.44


,Date,qc,Tc_in,Tcond,Tc_out,mv,Tv,Tamb,Cw
0,09-Feb-2024,16.00,25.13,33.315184,32.555759,205.61,36.03,14.89,88.248
1,26-Feb-2024,10.00,17.07,33.543341,32.487754,266.93,36.09,16.70,120.600
2,16-Apr-2024,22.43,33.12,39.294241,38.549930,212.22,43.30,25.06,0.000
3,16-Apr-2024,24.00,33.18,38.938294,38.187016,209.40,43.35,27.42,140.064
4,16-Apr-2024,23.74,33.16,39.689669,38.848472,235.36,43.52,29.84,124.710
5,16-Apr-2024,24.00,31.66,37.713483,36.907559,219.49,43.44,31.02,139.218
6,18-Apr-2024,24.00,33.17,39.530026,38.700143,231.26,43.28,19.55,124.800
7,18-Apr-2024,24.00,33.16,39.119698,38.341941,216.70,43.28,20.82,130.824
8,18-Apr-2024,24.00,33.90,41.289329,40.334418,269.14,43.55,22.24,0.000
9,18-Apr-2024,24.00,33.68,39.775340,38.985338,221.89,43.43,23.09,0.000


,Tv,qc,Tc_in,Tcond,Tc_out,mv
0,36.11,10.00,17.07,34.74,32.27,268.20
1,36.17,24.00,26.74,34.90,32.60,250.33
2,54.90,10.00,38.09,54.42,53.05,270.60
3,55.70,23.81,47.41,53.13,53.40,253.50
4,35.95,16.00,26.10,34.76,32.48,186.50
5,36.31,16.00,20.11,35.04,31.18,317.60
6,55.20,16.00,44.63,53.21,51.91,196.00
7,55.87,16.00,43.45,54.24,53.64,327.72
8,43.69,10.00,31.13,43.16,42.11,194.82
9,44.51,10.00,25.07,43.76,42.00,304.18


,Unnamed: 0,Tv,Tc_in,Tc_out,qc,mv,Tcond
0,0,36.11,21.803732,37.306268,10.00,268.20,36.11
1,1,36.17,26.097781,32.126439,24.00,250.33,36.17
2,2,54.90,37.365255,52.706281,10.00,270.60,54.90
3,3,55.70,47.521408,53.700000,23.81,253.50,55.70
4,4,35.95,26.819283,33.557881,16.00,186.50,35.95
5,5,36.31,23.919240,35.390760,16.00,317.60,36.31
6,6,55.20,46.210807,53.200000,16.00,196.00,55.20
7,7,55.87,41.921376,53.521011,16.00,327.72,55.87
8,8,43.69,30.488109,41.690000,10.00,194.82,43.69
9,9,44.51,25.073786,42.436214,10.00,304.18,44.51


In [19]:
from phd_visualizations.super_makers import SuperMarker

fig = regression_plot(
    df_ref= df, 
    df_mod= results_df,
    df_ref_bg= df_cal,
    df_mod_bg= results_cal_df,
    bg_label="Cal.",
    var_ids=["Tc_out", "Tcond"], 
    units=["℃"]*2, 
    instruments=[SupportedInstruments.pt100]*2,
    show_error_metrics=["r2", "mae"],
    var_labels=["T<sub>c,out</sub>", "T<sub>cond</sub>"],
    legend_pos="top_spaced",
    inline_error_metrics_text=True,
    
    # kwargs for layout
    title_text="<b>Condenser</b> model validation",
    legend_y = 1.06, 
    legend_x= 0.5,
    # legend_itemwidth = 100,
    title_y=0.99,
    # legend_font=dict(size=12),
    width=350,#530,
    height=800,
    vertical_spacing=0.15,
    margin=dict(l=10, r=10, t=130, b=10),
    
    # super_marker=SuperMarker(
    #     size_var_id="subcooling",
    #     size_var_range=[df["subcooling"].min(), df["subcooling"].max()],
    #     size_label="Max. Subcooling (~ 2ºC)",
    #     fill_label="Subcooling"
    # )
    
    # super_marker=SuperMarker(
    #     size_var_id="mv",
    #     size_var_range=[df.mv.min(), df.mv.max()],
    #     size_label="Vapor mass flow rate",
    # )
)
# fig.update_layout(legend_y = 1.1, legend_font=dict(size=12))

fig


In [20]:
save_figure(
    fig,
    figure_name="condenser_model_regression",
    figure_path=Path("../results"),
    formats=["png", "svg"],
    scale=2
)


2025-10-02 20:37:03.922 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/condenser_model_regression.png
2025-10-02 20:37:07.037 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/condenser_model_regression.svg
